# 03 — VLM JSONL (five-source evidence card)

Each line of `dataset.jsonl` is one patch:

- T0/T1 RGB (512×512)
- Sentinel-2 10-band optical + indices
- Sentinel-1 VV/VH
- CHIRPS rainfall
- ERA5-Land temperature & soil-moisture anomalies
- DEM elevation / slope / aspect
- Otsu triplets + explanation

**Lat/lon and place names are omitted** from the VLM-facing record (roadmap §5).


## 1. Load DIP records


In [1]:
from pathlib import Path
import json
import math
import sys

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.text_utils import generate_vlm_explanation

TRIPLETS_JSON = ROOT / "data" / "raw_metadata" / "triplets.json"
OUT_JSONL = ROOT / "dataset.jsonl"
SIDECAR = ROOT / "data" / "raw_metadata" / "explanations_log.json"
FINAL_LABELS = ROOT.parent / "data" / "labels" / "final_labels.csv"

records = json.loads(TRIPLETS_JSON.read_text())
final_labels = pd.read_csv(FINAL_LABELS) if FINAL_LABELS.exists() else pd.DataFrame()
ref_map = dict(zip(final_labels.patch_id.astype(str), final_labels.reference_class)) if not final_labels.empty else {}
print("scenes", len(records), "| reference labels", len(ref_map))


scenes 366 | reference labels 19


## 2. Build evidence cards and generate explanations


In [2]:
def _clean(obj):
    if isinstance(obj, dict):
        return {str(k): _clean(v) for k, v in obj.items() if k not in {"lon", "lat"}}
    if isinstance(obj, float) and (math.isnan(obj) or math.isinf(obj)):
        return None
    if hasattr(obj, "item"):
        try:
            return obj.item()
        except Exception:
            return str(obj)
    return obj


def supporting_context(rec):
    s = rec.get("scalars") or {}
    csv = rec.get("csv_stats") or {}
    merged = {**csv, **s}
    keys = [
        "ndvi_t0", "ndvi_t1", "nbr_t0", "nbr_t1",
        "vv_t0", "vv_t1", "vh_t0", "vh_t1",
        "chirps_rainfall_t0", "chirps_rainfall_t1",
        "era5_temp_anomaly_t0", "era5_temp_anomaly_t1",
        "era5_sm_anomaly_t0", "era5_sm_anomaly_t1",
        "elevation_m", "slope_deg", "aspect_deg",
        "masked_frac_t0", "masked_frac_t1",
    ]
    # parent CSV used *_before / *_after names
    alias = {
        "ndvi_before": "ndvi_t0", "ndvi_after": "ndvi_t1",
        "nbr_before": "nbr_t0", "nbr_after": "nbr_t1",
        "vv_before": "vv_t0", "vv_after": "vv_t1",
        "vh_before": "vh_t0", "vh_after": "vh_t1",
        "chirps_rainfall_before": "chirps_rainfall_t0",
        "chirps_rainfall_after": "chirps_rainfall_t1",
        "era5_temp_anomaly_before": "era5_temp_anomaly_t0",
        "era5_temp_anomaly_after": "era5_temp_anomaly_t1",
        "era5_sm_anomaly_before": "era5_sm_anomaly_t0",
        "era5_sm_anomaly_after": "era5_sm_anomaly_t1",
        "masked_frac_before": "masked_frac_t0",
        "masked_frac_after": "masked_frac_t1",
    }
    out = {}
    for src, dst in alias.items():
        if src in merged and dst not in merged:
            merged[dst] = merged[src]
    for k in keys:
        if k in merged:
            out[k] = merged[k]
    return _clean(out)


OLLAMA_MODEL = "llama3"
logs = []
dataset_rows = []

for rec in records:
    ctx = supporting_context(rec)
    result = generate_vlm_explanation(rec.get("triplets") or [], model=OLLAMA_MODEL, context=ctx)
    logs.append({"scene_id": rec["scene_id"], "source": result["source"], "model": result["model"], "error": result["error"]})
    p = rec["paths"]
    s = rec.get("scalars") or {}
    csv = rec.get("csv_stats") or {}
    merged = {**csv, **s}

    row = {
        "id": rec["scene_id"],
        "task": "ground_forest_disturbance_in_spectral_evidence",
        "vlm_image_size": rec.get("vlm_size", [512, 512]),
        "images": {"t0": p.get("t0_image"), "t1": p.get("t1_image")},
        "sentinel2": {
            "bands": ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"],
            "indices": ["NDVI", "EVI", "NDMI", "NBR"],
            "optical_t0": p.get("t0_optical"),
            "optical_t1": p.get("t1_optical"),
            "indices_t0": p.get("t0_indices"),
            "indices_t1": p.get("t1_indices"),
            "optical_t0_tif": p.get("t0_optical_tif"),
            "optical_t1_tif": p.get("t1_optical_tif"),
        },
        "sentinel1": {
            "polarizations": ["VV", "VH"],
            "t0": p.get("t0_s1"),
            "t1": p.get("t1_s1"),
            "t0_tif": p.get("t0_s1_tif"),
            "t1_tif": p.get("t1_s1_tif"),
            "orbit_pass": merged.get("s1_orbit_pass"),
            "n_scenes_t0": merged.get("s1_n_t0", merged.get("s1_n_before")),
            "n_scenes_t1": merged.get("s1_n_t1", merged.get("s1_n_after")),
        },
        "chirps": {
            "variable": "rainfall_mm",
            "t0": p.get("t0_rainfall"),
            "t1": p.get("t1_rainfall"),
            "t0_tif": p.get("t0_rainfall_tif"),
            "t1_tif": p.get("t1_rainfall_tif"),
            "rainfall_mm_t0": merged.get("chirps_rainfall_t0", merged.get("chirps_rainfall_before")),
            "rainfall_mm_t1": merged.get("chirps_rainfall_t1", merged.get("chirps_rainfall_after")),
        },
        "era5_land": {
            "variables": ["temperature_anomaly", "soil_moisture_anomaly"],
            "climatology": "2015-01-01/2021-12-31",
            "t0": p.get("t0_era5"),
            "t1": p.get("t1_era5"),
            "t0_tif": p.get("t0_era5_tif"),
            "t1_tif": p.get("t1_era5_tif"),
            "temp_anomaly_t0": merged.get("era5_temp_anomaly_t0", merged.get("era5_temp_anomaly_before")),
            "temp_anomaly_t1": merged.get("era5_temp_anomaly_t1", merged.get("era5_temp_anomaly_after")),
            "soil_moisture_anomaly_t0": merged.get("era5_sm_anomaly_t0", merged.get("era5_sm_anomaly_before")),
            "soil_moisture_anomaly_t1": merged.get("era5_sm_anomaly_t1", merged.get("era5_sm_anomaly_after")),
        },
        "dem": {
            "variables": ["elevation_m", "slope_deg", "aspect_deg"],
            "stack": p.get("dem"),
            "tif": p.get("dem_tif"),
            "elevation_m": merged.get("elevation_m"),
            "slope_deg": merged.get("slope_deg"),
            "aspect_deg": merged.get("aspect_deg"),
        },
        "evidence": {
            "delta_ndvi": p.get("delta_ndvi"),
            "delta_nbr": p.get("delta_nbr"),
            "delta_ndvi_tif": p.get("delta_ndvi_tif"),
            "delta_nbr_tif": p.get("delta_nbr_tif"),
            "delta_ndvi_mask": p.get("delta_ndvi_mask"),
            "delta_nbr_mask": p.get("delta_nbr_mask"),
        },
        "acquisition": {
            "season": "dry_season_jan_feb",
            "t0_window": [merged.get("t0_start", "2022-01-01"), merged.get("t0_end", "2022-02-28")],
            "t1_window": [merged.get("t1_start", "2023-01-01"), merged.get("t1_end", "2023-02-28")],
            "masked_fraction_t0": merged.get("masked_frac_t0", merged.get("masked_frac_before")),
            "masked_fraction_t1": merged.get("masked_frac_t1", merged.get("masked_frac_after")),
        },
        "triplets": [
            {"bbox": t[0], "disturbance_type": t[1], "severity": t[2]}
            for t in rec.get("triplets") or []
        ],
        "explanation": result["explanation"],
        "explanation_source": result["source"],
        "sources_present": rec.get("sources_present"),
        "candidate_class": merged.get("candidate_class"),
        "reference_class": ref_map.get(rec["scene_id"]),
    }
    dataset_rows.append(_clean(row))
    print(rec["scene_id"], result["source"], "boxes", len(row["triplets"]))

SIDECAR.write_text(json.dumps(logs, indent=2))


p00000 fallback boxes 0
p00028 fallback boxes 0
p00031 fallback boxes 0
p00044 fallback boxes 0
p00052 fallback boxes 2
p00086 fallback boxes 0
p00100 fallback boxes 0
p00109 fallback boxes 0
p00126 fallback boxes 0
p00133 fallback boxes 0
p00154 fallback boxes 0
p00166 fallback boxes 0
p00183 fallback boxes 0
p00264 fallback boxes 0
p00267 fallback boxes 0
p00279 fallback boxes 0
p00293 fallback boxes 0
p00296 fallback boxes 0
p00297 fallback boxes 0
p00298 fallback boxes 0
p00312 fallback boxes 0
p00325 fallback boxes 0
p00328 fallback boxes 0
p00339 fallback boxes 0
p00355 fallback boxes 0
p00374 fallback boxes 0
p00385 fallback boxes 0
p00402 fallback boxes 0
p00453 fallback boxes 0
p00466 fallback boxes 0
p00469 fallback boxes 0
p00491 fallback boxes 0
p00525 fallback boxes 0
p00564 fallback boxes 0
p00575 fallback boxes 0
p00587 fallback boxes 0
p00593 fallback boxes 0
p00631 fallback boxes 0
p00641 fallback boxes 0
p00657 fallback boxes 0
p00662 fallback boxes 0
p00701 fallback 

139082

## 3. Write dataset.jsonl


In [3]:
with OUT_JSONL.open("w", encoding="utf-8") as f:
    for row in dataset_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
print(f"Wrote {len(dataset_rows)} records -> {OUT_JSONL}")
preview = json.loads(OUT_JSONL.read_text().splitlines()[0])
preview["explanation"] = (preview.get("explanation") or "")[:240]
print(json.dumps({k: preview[k] for k in ["id", "sentinel2", "sentinel1", "chirps", "era5_land", "dem", "triplets", "explanation"] if k in preview}, indent=2)[:4000])


Wrote 366 records -> /Users/vaibhav/Desktop/Independent Study/is work/DataSet Extractor/ecoexplain_vlm_dataset/dataset.jsonl
{
  "id": "p00000",
  "sentinel2": {
    "bands": [
      "B2",
      "B3",
      "B4",
      "B5",
      "B6",
      "B7",
      "B8",
      "B8A",
      "B11",
      "B12"
    ],
    "indices": [
      "NDVI",
      "EVI",
      "NDMI",
      "NBR"
    ],
    "optical_t0": "data/processed_images/p00000_t0_optical.npy",
    "optical_t1": "data/processed_images/p00000_t1_optical.npy",
    "indices_t0": "data/processed_images/p00000_t0_indices.npy",
    "indices_t1": "data/processed_images/p00000_t1_indices.npy",
    "optical_t0_tif": "data/raw_tiffs/p00000_T0_s2_optical.tif",
    "optical_t1_tif": "data/raw_tiffs/p00000_T1_s2_optical.tif"
  },
  "sentinel1": {
    "polarizations": [
      "VV",
      "VH"
    ],
    "t0": "data/processed_images/p00000_t0_s1.npy",
    "t1": "data/processed_images/p00000_t1_s1.npy",
    "t0_tif": "data/raw_tiffs/p00000_T0_s1.tif",


## 4. Completeness check


In [4]:
needed_blocks = ["images", "sentinel2", "sentinel1", "chirps", "era5_land", "dem", "evidence", "triplets", "explanation"]
s2_bands = ["B2", "B3", "B4", "B5", "B6", "B7", "B8", "B8A", "B11", "B12"]
errors = []
for row in dataset_rows:
    for b in needed_blocks:
        if b not in row:
            errors.append((row["id"], b))
    if row["sentinel2"].get("bands") != s2_bands:
        errors.append((row["id"], "s2_band_list"))
    if row["sentinel1"].get("polarizations") != ["VV", "VH"]:
        errors.append((row["id"], "s1_pol"))
print("schema errors:", errors or "none")
print("records", len(dataset_rows))
print("ollama/fallback", {src: sum(1 for r in dataset_rows if r["explanation_source"] == src) for src in ("ollama", "fallback")})


schema errors: none
records 366
ollama/fallback {'ollama': 0, 'fallback': 366}
